In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
import joblib

In [12]:
df = pd.read_csv('/home/nursss/Загрузки/Smart_irrigation_dataset.csv')


In [26]:
class SmartIrrigationModel:
    def __init__(self):
        # Шаг 1: Классификатор (Нужен ли полив?)
        self.classifier = lgb.LGBMClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=5,
            class_weight='balanced', random_state=42
        )
        # Шаг 2: Регрессор (Сколько поливать?)
        self.regressor = xgb.XGBRegressor(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, random_state=42
        )
        self.feature_columns = None

    def feature_engineering(self, df):
        """Создает новые признаки на основе физики процесса"""
        data = df.copy()

        # Дефицит влаги (мм)
        data['moisture_deficit_mm'] = ((data['field_capacity_%'] - data['soil_moisture_%']) / 100) * (data['root_zone_depth_m'] * 1000)
        data['moisture_deficit_mm'] = data['moisture_deficit_mm'].clip(lower=0)

        # Потребность культуры (ETc)
        data['ETc_mm_day'] = data['reference_evapotranspiration_ET0_mm_day'] * data['crop_coefficient_Kc']

        # Влияние КПД системы
        data['efficiency_multiplier'] = 100.0 / data['application_efficiency_%']

        # Цикличность времени года
        data['day_sin'] = np.sin(data['day_of_year'] * (2 * np.pi / 365))
        data['day_cos'] = np.cos(data['day_of_year'] * (2 * np.pi / 365))

        # Кодирование категорий (Культура, Тип почвы)
        data = pd.get_dummies(data, columns=['crop_name', 'soil_type'], drop_first=True)
        return data

    def train(self, df):
        """Обучает модель и запоминает структуру колонок"""
        print("⏳ Начинаем обучение модели...")
        data = self.feature_engineering(df)

        y_class = data['irrigate']
        y_reg = data['irrigation_amount_m3']

        # Удаляем таргеты и ID из признаков
        cols_to_drop = ['irrigate', 'irrigation_amount_m3']
        if 'Unnamed: 0' in data.columns: cols_to_drop.append('Unnamed: 0')

        X = data.drop(columns=cols_to_drop)
        self.feature_columns = X.columns # ЗАПОМИНАЕМ КОЛОНКИ ДЛЯ ПРОДАКШЕНА

        # Обучение классификатора
        self.classifier.fit(X, y_class)

        # Обучение регрессора (только на тех данных, где полив был нужен)
        mask = (y_class == 1)
        self.regressor.fit(X[mask], y_reg[mask])
        print("✅ Обучение завершено!")

    def predict(self, input_data):
        """
        Универсальная функция предсказания.
        Принимает либо словарь (одна запись), либо DataFrame (несколько записей).
        """
        # 1. Проверяем тип входных данных и превращаем в DataFrame
        if isinstance(input_data, dict):
            df_new = pd.DataFrame([input_data])
        elif isinstance(input_data, pd.DataFrame):
            df_new = input_data.copy()
        else:
            raise ValueError("Входные данные должны быть словарем (dict) или pandas DataFrame")

        # 2. Применяем генерацию признаков
        X_new = self.feature_engineering(df_new)

        # 3. Выравниваем колонки (защита от отсутствующих категорий)
        X_new = X_new.reindex(columns=self.feature_columns, fill_value=0)

        # 4. Шаг 1: Решение о поливе (массив из 0 и 1)
        should_irrigate = self.classifier.predict(X_new)

        # 5. Шаг 2: Объем полива (массив объемов)
        predicted_volumes = self.regressor.predict(X_new)

        # Убираем возможные отрицательные значения от регрессора
        predicted_volumes = np.maximum(0.0, predicted_volumes)

        # Итоговый объем: умножаем предсказание объема на решение (0 или 1)
        final_volumes = should_irrigate * predicted_volumes

        # 6. Форматируем вывод в зависимости от того, что было на входе
        if isinstance(input_data, dict):
            # Если был словарь, возвращаем красивый JSON-подобный ответ
            return {
                "needs_irrigation": bool(should_irrigate[0]),
                "irrigation_volume_m3": round(float(final_volumes[0]), 2)
            }
        else:
            # Если был DataFrame, возвращаем два массива (объемы и решения)
            return np.round(final_volumes, 2), should_irrigate

    def save(self, file_path='smart_irrigation_model.pkl'):
        """Сохраняет модель на жесткий диск"""
        joblib.dump(self, file_path)
        print(f"💾 Модель сохранена в: {file_path}")

    @classmethod
    def load(cls, file_path='smart_irrigation_model.pkl'):
        """Загружает модель с жесткого диска"""
        model = joblib.load(file_path)
        print(f"📂 Модель загружена из: {file_path}")
        return model

In [27]:
ai_farmer = SmartIrrigationModel()
ai_farmer.train(df)

⏳ Начинаем обучение модели...
[LightGBM] [Info] Number of positive: 5538, number of negative: 4462
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001822 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5425
[LightGBM] [Info] Number of data points in the train set: 10000, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furt

In [28]:
if __name__ == "__main__":

    # ---------------------------------------------------------
    # ЭТАП 1: ОБУЧЕНИЕ И СОХРАНЕНИЕ (Делается один раз)
    # ---------------------------------------------------------
    print("\n--- ЭТАП 1: ОБУЧЕНИЕ ---")
    # Загружаем ваш CSV файл
    # Замените 'irrigation_data.csv' на реальное имя вашего файла
    df_history = pd.read_csv('/home/nursss/Загрузки/Smart_irrigation_dataset.csv')

    # Инициализируем, обучаем и сохраняем
    ai_model = SmartIrrigationModel()
    ai_model.train(df_history)
    ai_model.save('production_model_v1.pkl')
new_data = df.sample(10).drop(columns=['irrigate', 'irrigation_amount_m3']) # Имитация новых данных без таргетов
volumes, decisions = ai_farmer.predict(new_data)
print("\nРешения о поливе (0/1):", decisions)
print("Объемы полива (м3):", np.round(volumes, 2))


--- ЭТАП 1: ОБУЧЕНИЕ ---
⏳ Начинаем обучение модели...
[LightGBM] [Info] Number of positive: 5538, number of negative: 4462
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001143 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5425
[LightGBM] [Info] Number of data points in the train set: 10000, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

In [29]:
# ---------------------------------------------------------
# ЭТАП 2: ИСПОЛЬЗОВАНИЕ В ПРОДАКШЕНЕ (Вызывается по запросу)
# ---------------------------------------------------------
print("\n--- ЭТАП 2: ПРЕДСКАЗАНИЕ (INFERENCE) ---")

# 1. Загружаем готовую модель
prod_model = SmartIrrigationModel.load('production_model_v1.pkl')

# 2. Получаем данные с датчиков (ОДНА СТРОКА, БЕЗ 'irrigate' и 'irrigation_amount_m3')
sensor_data_today = {
    'crop_name': 'Tomato',
    'soil_type': 'Loamy',
    'crop_age_days': 58,
    'day_of_year': 176,
    'temperature_C': 32.24,
    'humidity_%': 60.81,
    'rainfall_mm': 0.0,
    'effective_rainfall_mm': 0.0,
    'solar_radiation_MJ_m2_day': 18.35,
    'wind_speed_m_s': 1.53,
    'field_capacity_%': 30.25,
    'wilting_point_%': 19.59,
    'soil_moisture_%': 20.10,
    'reference_evapotranspiration_ET0_mm_day': 3.061,
    'crop_coefficient_Kc': 1.027,
    'application_efficiency_%': 78.36,
    'root_zone_depth_m': 0.63,
    'available_water_content_mm_per_m': 176.37,
    'irrigation_interval_days': 7,
    'p_fraction': 0.477
}

# 3. Делаем предсказание для одной строки (словаря)
result = prod_model.predict(sensor_data_today)

print("\n📊 РЕЗУЛЬТАТ ИИ ДЛЯ ОДНОГО ПОЛЯ:")
if result['needs_irrigation']:
    print(f"💧 ВНИМАНИЕ: Требуется полив! Рекомендуемый объем: {result['irrigation_volume_m3']} куб.м.")
else:
    print("☀️ Полив не требуется. Почва достаточно увлажнена.")

# ---------------------------------------------------------
# 4. Тестируем на нескольких строках (DataFrame)
# ---------------------------------------------------------
print("\n--- ПРЕДСКАЗАНИЕ ДЛЯ ПАКЕТА ДАННЫХ (DATAFRAME) ---")
# Берем 5 случайных строк из оригинального датасета и удаляем таргеты
new_data_df = df.sample(5).drop(columns=['irrigate', 'irrigation_amount_m3'])

# Передаем DataFrame в функцию predict
volumes, decisions = prod_model.predict(new_data_df)

print("Решения о поливе (0/1):", decisions)
print("Объемы полива (м3):", volumes)


--- ЭТАП 2: ПРЕДСКАЗАНИЕ (INFERENCE) ---
📂 Модель загружена из: production_model_v1.pkl

📊 РЕЗУЛЬТАТ ИИ ДЛЯ ОДНОГО ПОЛЯ:
💧 ВНИМАНИЕ: Требуется полив! Рекомендуемый объем: 338.84 куб.м.

--- ПРЕДСКАЗАНИЕ ДЛЯ ПАКЕТА ДАННЫХ (DATAFRAME) ---
Решения о поливе (0/1): [0 1 0 1 0]
Объемы полива (м3): [  0.   567.13   0.   258.82   0.  ]
